In [1]:
%pip install -r requirements.txt

ERROR: Could not find a version that satisfies the requirement anaconda-anon-usage (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for anaconda-anon-usage
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import json
from chunking import SectionAwareChunker, SemanticChunker
from embeddings import MiniLMEmbedding, QwenEmbedding
from vectore_store import FAISSVectorDB
from ranking_n_retrieval import Retriever
from llm_n_prompt import QwenLLM, PromptTemplate
from rag_pipeline import RAGPipeline
from evaluation import evaluate_rag_pipeline

/Users/manish09/Desktop/ttm/CuisineRAG/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def build_embedder(choice):
    if choice == "minilm":
        return MiniLMEmbedding(), 384
    elif choice == "qwen":
        return QwenEmbedding(), 1024
    else:
        raise ValueError(f"Unknown embedder: {choice!r}. Choose 'minilm' or 'qwen'")


def build_vectordb(dim):
    return FAISSVectorDB(dim=dim)


def build_retriever(choice, vectordb, documents):
    retriever = Retriever(vectordb, documents)
    retriever.active_combo = choice
    return retriever

In [4]:
def run_json_input_output(input_file_name, output_file_name):
    import os

    # --- build components ---
    if CHUNKER == "semantic":
        chunker = SemanticChunker()
    else:
        chunker = SectionAwareChunker()

    embedder, dim = build_embedder(EMBEDDING)
    vectordb       = build_vectordb(dim)
    prompt_builder = PromptTemplate()
    llm            = QwenLLM(device=DEVICE)

    pipeline = RAGPipeline(
        chunker        = chunker,
        embedder       = embedder,
        vectordb       = vectordb,
        retriever      = None,
        prompt_builder = prompt_builder,
        llm            = llm,
    )

    # --- FAISS cache: one file-pair per (embedder, chunker) combination ---
    INDEX_BIN = f"faiss_index_{EMBEDDING}_{CHUNKER}.bin"
    DOCS_JSON  = f"faiss_docs_{EMBEDDING}_{CHUNKER}.json"

    if os.path.exists(INDEX_BIN) and os.path.exists(DOCS_JSON):
        vectordb.load(INDEX_BIN, DOCS_JSON)
        pipeline.chunks = vectordb.documents
        print(f"Loaded existing FAISS index from disk: {INDEX_BIN}")
    else:
        pipeline.index_data(FILEPATHS)
        vectordb.save(INDEX_BIN, DOCS_JSON)

    # --- build retriever with chunks for BM25 ---
    retriever = build_retriever(RETRIEVAL, vectordb, pipeline.chunks)
    pipeline.retriever = retriever

    with open(input_file_name) as input_file:
        query_list = json.load(input_file)["queries"]

    print(f"\nProcessing {len(query_list)} queries from {input_file_name}...\n")

    results = [0] * len(query_list)
    for i, query in enumerate(query_list):
        query_id   = query["query_id"]
        query_text = query["query"].strip()
        print(f"[{i+1}/{len(query_list)}] {query_text[:60]}...")
        answer, docs = pipeline.query(query_text)
        results[i] = {
            "query_id": str(query_id),
            "query":    str(query_text),
            "response": str(answer),
            "retrieved_context": [
                {"doc_id": chunk.metadata.get("doc_id", "?"), "text": chunk.page_content}
                for chunk in docs
            ],
        }

    final = {"results": results}
    with open(output_file_name, "w") as output_json:
        json.dump(final, output_json, indent=2, ensure_ascii=False)

    print(f"\nDone! {len(results)} results saved to {output_file_name}")

In [5]:
# ==============================================================
# CONFIGURATION — change these to switch models / retrieval
# ==============================================================
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"

CHUNKER   = "semantic"  # "section" or "semantic"
EMBEDDING = "minilm"      # "minilm"  or "qwen"
RETRIEVAL = "combo1"    # "combo1"  or "combo2"
DEVICE    = get_device()
FILEPATHS = [
    "data/raw/south_asian_corpus.json",
    "data/raw/saved_wikibook_data.json",
    "data/raw/cuisines80.json",
]

Please put your input .json file in the **inputs_and_outputs** folder, this folder will also contain your output after you run the 
**run_json_input_output()** function.

In [6]:
input_file_path = "inputs_and_outputs/input_payload_sample.json" #change as per your file name
output_file_path = "inputs_and_outputs/output_payload_sample.json" #change if needed or leave as is

In [7]:
if __name__ == "__main__":
    run_json_input_output(input_file_path, output_file_path)

Loading embedding model for semantic chunking (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8750.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading MiniLM embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13247.47it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initializing FAISS...
Loading LLM...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4468.89it/s]


FAISS loaded ← faiss_index_minilm_semantic.bin (12,708 chunks, dim=384)
Loaded existing FAISS index from disk: faiss_index_minilm_semantic.bin
Loading cross-encoder reranker...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6824.55it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Processing 14 queries from inputs_and_outputs/input_payload_sample.json...

[1/14] What is Dosa and where does it originate from?...
[2/14] What is Samosas and where does it originate from?...
[3/14] What is Chicken Tikka Masala and where is it from?...
[4/14] What is a biryani?...
[5/14] What is anglo-Indian food?...
[6/14] What is Pongal?...
[7/14] What is Garam Masala?...
[8/14] What is upma?...
[9/14] What is ghevar usually served with?...
[10/14] What is Nihari?...
[11/14] What is Kashmiri Pink Tea?...
[12/14] What is the main fish used in Bengali cuisine?...
[13/14] What are Sri Lankan Hoppers?...
[14/14] What is Chana Dal Halwa?...

Done! 14 results saved to inputs_and_outputs/output_payload_sample.json


### EVALUATION PIPELINE

In [8]:
# Replace these filenames with the actual paths to your JSON files
eval_df = evaluate_rag_pipeline(
        output_path='inputs_and_outputs/output_payload_sample.json', 
        benchmark_path='data/benchmark_dataset.json'
    )

  RAG PIPELINE EVALUATION

[Setup] Loading models …


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17851.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 5803.54it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTE

[Setup] Done.



Evaluating queries: 100%|███████████████████████| 14/14 [00:03<00:00,  3.55it/s]


  RESULTS SUMMARY

── GENERATION METRICS ──────────────────────────────────────
  BLEU (nltk)          : 0.0804
  ROUGE-1 F1           : 0.2960
  ROUGE-1 Precision    : 0.2203
  ROUGE-1 Recall       : 0.5726
  SacreBLEU            : 8.47
  ChrF3++              : 37.31
  BERTScore F1         : 0.8818

── RETRIEVAL METRICS ───────────────────────────────────────
  Keyword Recall       : 66.34%
  Semantic Similarity  : 0.7083
  Precision            : 0.0000
  Recall               : 0.0000
  MRR                  : 0.0000

